# Module 01 — Notebook 03: Saliency Deep Dive

## Learning Objectives
By completing this notebook, you will:
1. Demonstrate the gradient saturation problem empirically in a controlled setting
2. Quantify the noise problem in saliency maps using gradient variance
3. Show how SmoothGrad reduces gradient noise while preserving signal
4. Understand why saliency fails in regions near ReLU saturation boundaries
5. Build the intuition that motivates Integrated Gradients (Module 02)

## Prerequisites
- Notebooks 01 and 02 from this module
- Guide 01 (gradient theory) from this module

## Estimated Time: 14 minutes

---

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
import io
import json
from PIL import Image

from captum.attr import Saliency, IntegratedGradients, NoiseTunnel
from captum.attr import visualization as viz

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

def denormalize(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)
    return torch.clamp(t.cpu() * std + mean, 0, 1)

# Load ResNet-50
model = models.resnet50(weights='IMAGENET1K_V1').to(DEVICE)
model.eval()
print("ResNet-50 loaded.")

## 1. The Saturation Problem: Controlled Demonstration

Gradient saturation occurs when a neuron's input is in the flat (zero-gradient) region of a ReLU. We demonstrate this by:

1. Building a minimal network with a controlled saturation regime
2. Showing that saliency gives zero attribution to a feature that provably affects the output
3. Showing that Integrated Gradients gives the correct non-zero attribution

In [ ]:
# Minimal network to demonstrate saturation
# Architecture: 2 inputs → ReLU → 1 output
# Weights chosen so one input is in the saturation zone

class SaturationDemo(nn.Module):
    """
    Simple network for demonstrating gradient saturation.
    f(x1, x2) = ReLU(w1*x1 + w2*x2 + b)
    With fixed weights, we control which input is in the saturation region.
    """
    def __init__(self, w1, w2, bias):
        super().__init__()
        self.layer = nn.Linear(2, 1, bias=True)
        # Set weights manually for controlled experiment
        with torch.no_grad():
            self.layer.weight.copy_(torch.tensor([[w1, w2]]))
            self.layer.bias.copy_(torch.tensor([bias]))
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.layer(x))


# Set up: w1=2, w2=1, bias=-5
# f(x1, x2) = ReLU(2*x1 + x2 - 5)
sat_model = SaturationDemo(w1=2.0, w2=1.0, bias=-5.0)
sat_model.eval()

# Choose an input in the saturation zone for x1 but not x2
# At x = (1.0, 0.5): 2*1.0 + 0.5 - 5 = -2.5 → ReLU → 0
# The gradient is ZERO (inactive ReLU)
# But if we increase x1 from 1.0 to 2.5: 2*2.5 + 0.5 - 5 = 0.5 → ReLU → 0.5
# So x1 IS relevant — changing it changes the output!

x_sat = torch.tensor([[1.0, 0.5]])  # In saturation zone
x_active = torch.tensor([[3.0, 0.5]])  # Out of saturation zone

print("Saturation Demo")
print("=" * 50)
print(f"Network: f(x1, x2) = ReLU(2*x1 + x2 - 5)")
print()

with torch.no_grad():
    out_sat    = sat_model(x_sat).item()
    out_active = sat_model(x_active).item()

print(f"Saturated input x = (1.0, 0.5):")
print(f"  Pre-ReLU: 2*1.0 + 0.5 - 5 = -2.5")
print(f"  f(x) = ReLU(-2.5) = {out_sat}")
print(f"  Gradient: 0 (ReLU inactive)")
print()
print(f"Active input x = (3.0, 0.5):")
print(f"  Pre-ReLU: 2*3.0 + 0.5 - 5 = +1.5")
print(f"  f(x) = ReLU(1.5) = {out_active}")
print(f"  Gradient: 2.0 (ReLU active)")

# Demonstrate: x1 IS relevant to the output
# Moving from x1=1.0 to x1=3.0 changes output from 0 to 1.5
print()
print("x1 is RELEVANT: moving from 1.0 to 3.0 changes output from 0 to 1.5")
print("But gradient at (1.0, 0.5) = 0 — saliency says x1 has ZERO importance")
print("This is the saturation failure.")

In [ ]:
# Quantify the failure: Saliency vs Integrated Gradients on saturation demo

# For Captum, we need a 2-output model (for target class)
class SatDemoWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        out = self.model(x)  # (batch, 1)
        return torch.cat([-out, out], dim=-1)  # (batch, 2)

sat_wrapper = SatDemoWrapper(sat_model).eval()

# Saliency attributions
sal = Saliency(sat_wrapper)
x_sat_attr = x_sat.clone().requires_grad_(True)
sal_attr_sat = sal.attribute(x_sat_attr, target=1)

x_active_attr = x_active.clone().requires_grad_(True)
sal_attr_active = sal.attribute(x_active_attr, target=1)

# Integrated Gradients attributions (baseline = zero)
ig = IntegratedGradients(sat_wrapper)
baseline = torch.zeros(1, 2)

x_sat_attr2 = x_sat.clone().requires_grad_(True)
ig_attr_sat = ig.attribute(x_sat_attr2, baselines=baseline, target=1, n_steps=300)

x_active_attr2 = x_active.clone().requires_grad_(True)
ig_attr_active = ig.attribute(x_active_attr2, baselines=baseline, target=1, n_steps=300)

print("Saturation Failure Demonstration")
print("=" * 60)
print()
print("SATURATED INPUT x = (1.0, 0.5) → f(x) = 0")
print("-" * 40)
print(f"  Saliency:             x1={sal_attr_sat[0,0].item():.4f}, x2={sal_attr_sat[0,1].item():.4f}")
print(f"  Integrated Gradients: x1={ig_attr_sat[0,0].item():.4f}, x2={ig_attr_sat[0,1].item():.4f}")
print(f"  Ground truth: x1 is relevant, x2 is relevant")
print(f"  → Saliency gives ZERO for both (saturation failure)")
print(f"  → IG gives ZERO for both (output = 0, so attribution sums to 0 correctly)")
print()
print("ACTIVE INPUT x = (3.0, 0.5) → f(x) = 1.5")
print("-" * 40)
print(f"  Saliency:             x1={sal_attr_active[0,0].item():.4f}, x2={sal_attr_active[0,1].item():.4f}")
print(f"  Integrated Gradients: x1={ig_attr_active[0,0].item():.4f}, x2={ig_attr_active[0,1].item():.4f}")
print(f"  Ground truth: w1=2.0, w2=1.0, so x1 should be 2× more important")
print(f"  → Both Saliency and IG get this right when the ReLU is active")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

methods_sat = {
    'Saliency': sal_attr_sat.detach().numpy().flatten(),
    'IG': ig_attr_sat.detach().numpy().flatten()
}
methods_act = {
    'Saliency': sal_attr_active.detach().numpy().flatten(),
    'IG': ig_attr_active.detach().numpy().flatten()
}

for ax, (data, title) in zip(axes, [
    (methods_sat, "Saturated Input (f=0)\nSaliency fails — both features get 0"),
    (methods_act, "Active Input (f=1.5)\nBoth methods succeed"),
]):
    x_pos = np.arange(2)
    width = 0.3
    for j, (name, vals) in enumerate(data.items()):
        ax.bar(x_pos + j*width, np.abs(vals), width, label=name, alpha=0.8)
    ax.set_xticks(x_pos + width/2)
    ax.set_xticklabels(['x1 (weight=2)', 'x2 (weight=1)'])
    ax.set_ylabel('|Attribution|')
    ax.set_title(title)
    ax.legend()

plt.suptitle('Gradient Saturation: Saliency vs Integrated Gradients', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Gradient Noise in Deep Networks

In deep networks with many ReLU layers, the gradient at a specific input point is extremely sensitive to small perturbations. We quantify this by computing gradients at slightly perturbed versions of the same image.

In [ ]:
# Load a sample image
IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg"

try:
    with urllib.request.urlopen(IMAGE_URL, timeout=10) as resp:
        raw = Image.open(io.BytesIO(resp.read())).convert('RGB')
except Exception:
    arr = np.random.randint(80, 180, (320, 320, 3), dtype=np.uint8)
    raw = Image.fromarray(arr)

input_tensor = preprocess(raw).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    probs = torch.softmax(model(input_tensor), dim=1)
    top_class = probs.argmax().item()

# Load labels for display
LABELS_URL = (
    "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels"
    "/master/imagenet-simple-labels.json"
)
try:
    with urllib.request.urlopen(LABELS_URL) as resp:
        labels = json.loads(resp.read().decode())
except Exception:
    labels = [f"class_{i}" for i in range(1000)]
    labels[208] = "Labrador retriever"

print(f"Image loaded. Prediction: {labels[top_class]} ({probs.max():.1%})")

# Compute gradient noise: variance across perturbed inputs
N_PERTURBATIONS = 30
NOISE_STD = 0.05  # Small noise (5% of input range)

sal = Saliency(model)
perturbed_attributions = []

for i in range(N_PERTURBATIONS):
    # Add small Gaussian noise
    noise = torch.randn_like(input_tensor) * NOISE_STD
    perturbed = (input_tensor + noise).requires_grad_(True)
    attr = sal.attribute(perturbed, target=top_class).detach().cpu()
    perturbed_attributions.append(attr.squeeze(0).permute(1, 2, 0).abs().mean(-1).numpy())

# Stack and compute statistics
perturbed_stack = np.stack(perturbed_attributions, axis=0)  # (N, H, W)
attr_mean = perturbed_stack.mean(axis=0)    # Average attribution across perturbations
attr_std  = perturbed_stack.std(axis=0)     # Standard deviation (= noise level)
attr_cv   = attr_std / (attr_mean + 1e-8)  # Coefficient of variation

# Original attribution (no noise)
orig_attr = sal.attribute(
    input_tensor.requires_grad_(True), target=top_class
).detach().cpu().squeeze(0).permute(1, 2, 0).abs().mean(-1).numpy()

# Normalize for display
def norm99(arr):
    v = np.percentile(arr, 99)
    return np.clip(arr, 0, v) / (v + 1e-8)

img_np = denormalize(input_tensor.squeeze(0).cpu()).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(img_np)
axes[0].set_title("Original Image")
axes[0].axis('off')

im1 = axes[1].imshow(norm99(orig_attr), cmap='hot')
axes[1].set_title("Saliency (no noise)")
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(norm99(attr_std), cmap='Blues')
axes[2].set_title(f"Saliency Std Dev\nacross {N_PERTURBATIONS} noise samples\n(noise std={NOISE_STD})")
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

im3 = axes[3].imshow(norm99(attr_cv), cmap='YlOrRd')
axes[3].set_title("Coefficient of Variation\n(std/mean — relative noise)")
axes[3].axis('off')
plt.colorbar(im3, ax=axes[3], fraction=0.046)

plt.suptitle(
    f"Saliency Gradient Noise Analysis — {N_PERTURBATIONS} perturbations, σ={NOISE_STD}",
    fontsize=13
)
plt.tight_layout()
plt.show()

median_cv = np.median(attr_cv)
print(f"\nMedian coefficient of variation: {median_cv:.2f}")
print(f"  CV > 0.5 means the attribution varies by more than 50% across small perturbations.")
print(f"  This quantifies gradient noise: how unreliable is the attribution at any single point?")

## 3. SmoothGrad: Noise Reduction via Averaging

SmoothGrad (NoiseTunnel in Captum) computes the mean gradient over many noisy copies of the input. This reduces the high-frequency noise while preserving the spatially coherent signal.

In [ ]:
# Compare Saliency, SmoothGrad-Saliency, and Integrated Gradients
# on the same image to show noise reduction

# Method 1: Vanilla Saliency (baseline)
inp_sal = input_tensor.clone().requires_grad_(True)
sal_attr = Saliency(model).attribute(inp_sal, target=top_class)
sal_np = sal_attr.squeeze(0).cpu().permute(1,2,0).detach().abs().mean(-1).numpy()

# Method 2: SmoothGrad (NoiseTunnel wrapping Saliency)
nt_sal = NoiseTunnel(Saliency(model))
inp_smooth = input_tensor.clone().requires_grad_(True)
smooth_attr = nt_sal.attribute(
    inp_smooth,
    nt_type='smoothgrad',  # Average over noisy samples
    nt_samples=20,          # 20 noise samples
    stdevs=0.05,            # Noise std (same as above)
    target=top_class
)
smooth_np = smooth_attr.squeeze(0).cpu().permute(1,2,0).detach().abs().mean(-1).numpy()

# Method 3: VarGrad (variance across noise samples)
inp_var = input_tensor.clone().requires_grad_(True)
var_attr = nt_sal.attribute(
    inp_var,
    nt_type='vargrad',   # Variance (not mean) of attributions
    nt_samples=20,
    stdevs=0.05,
    target=top_class
)
var_np = var_attr.squeeze(0).cpu().permute(1,2,0).detach().abs().mean(-1).numpy()

# Method 4: Integrated Gradients (Module 02's method, shown for comparison)
ig = IntegratedGradients(model)
baseline = torch.zeros_like(input_tensor)
inp_ig = input_tensor.clone().requires_grad_(True)
ig_attr, delta = ig.attribute(
    inp_ig, baselines=baseline, target=top_class,
    n_steps=50, return_convergence_delta=True
)
ig_np = ig_attr.squeeze(0).cpu().permute(1,2,0).detach().abs().mean(-1).numpy()

print(f"IG convergence delta: {delta.item():.5f}")

# Visualize all four methods
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle(
    f"Noise Reduction Comparison — Prediction: {labels[top_class]}",
    fontsize=13
)

methods_data = [
    ("Saliency (1 pass)", sal_np, "1 backward pass"),
    ("SmoothGrad (20 avg)", smooth_np, "20 avg, noise σ=0.05"),
    ("VarGrad (20 var)", var_np, "20 var, noise σ=0.05"),
    ("Integrated Gradients", ig_np, "50 steps, zero baseline"),
]

for col, (name, attr_data, subtitle) in enumerate(methods_data):
    # Row 0: Heatmap
    im = axes[0, col].imshow(norm99(attr_data), cmap='hot', vmin=0, vmax=1)
    axes[0, col].set_title(f"{name}\n({subtitle})", fontsize=9)
    axes[0, col].axis('off')
    plt.colorbar(im, ax=axes[0, col], fraction=0.046)

    # Row 1: Overlay
    axes[1, col].imshow(img_np)
    axes[1, col].imshow(
        norm99(attr_data), alpha=0.6, cmap='hot',
        vmin=np.percentile(norm99(attr_data), 75)
    )
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Quantify noise reduction: compute gradient stability metric
# Stability = mean correlation between attribution maps for slightly perturbed inputs
# Higher stability = less noise sensitivity

print("Measuring attribution stability under small perturbations...")

def measure_stability(model_or_method, input_tensor, target_class,
                       n_trials=10, noise_std=0.03, is_noise_tunnel=False):
    """
    Measure attribution stability by computing correlations
    across perturbed inputs.

    Returns mean pairwise correlation across trials.
    """
    attrs = []
    for _ in range(n_trials):
        noise = torch.randn_like(input_tensor) * noise_std
        inp = (input_tensor + noise).requires_grad_(True)
        if is_noise_tunnel:
            attr = model_or_method.attribute(
                inp, nt_type='smoothgrad', nt_samples=10,
                stdevs=noise_std, target=target_class
            )
        else:
            attr = model_or_method.attribute(inp, target=target_class)
        mag = attr.squeeze(0).cpu().permute(1,2,0).detach().abs().mean(-1).numpy().flatten()
        attrs.append(mag)

    # Pairwise correlations
    corrs = []
    for i in range(n_trials):
        for j in range(i+1, n_trials):
            c = np.corrcoef(attrs[i], attrs[j])[0, 1]
            corrs.append(c)
    return np.mean(corrs)

# Measure stability for each method
methods_stability = {
    'Saliency':   (Saliency(model), False),
    'SmoothGrad': (NoiseTunnel(Saliency(model)), True),
}

stability_results = {}
for name, (method_obj, is_nt) in methods_stability.items():
    stab = measure_stability(method_obj, input_tensor, top_class,
                              n_trials=8, noise_std=0.03,
                              is_noise_tunnel=is_nt)
    stability_results[name] = stab
    print(f"  {name}: mean pairwise correlation = {stab:.3f}")

# Also measure IG stability
ig_obj = IntegratedGradients(model)
baseline = torch.zeros_like(input_tensor)

ig_attrs = []
for _ in range(8):
    noise = torch.randn_like(input_tensor) * 0.03
    inp = (input_tensor + noise).requires_grad_(True)
    attr = ig_obj.attribute(inp, baselines=baseline, target=top_class, n_steps=30)
    mag = attr.squeeze(0).cpu().permute(1,2,0).detach().abs().mean(-1).numpy().flatten()
    ig_attrs.append(mag)

ig_corrs = [np.corrcoef(ig_attrs[i], ig_attrs[j])[0,1]
            for i in range(8) for j in range(i+1, 8)]
stability_results['Integrated Gradients'] = np.mean(ig_corrs)
print(f"  Integrated Gradients: mean pairwise correlation = {stability_results['Integrated Gradients']:.3f}")

# Plot stability comparison
fig, ax = plt.subplots(figsize=(7, 4))
names = list(stability_results.keys())
values = list(stability_results.values())
colors = ['#e74c3c', '#3498db', '#2ecc71']
ax.bar(names, values, color=colors, alpha=0.8, edgecolor='white')
ax.set_ylabel('Mean Pairwise Correlation')
ax.set_ylim(0, 1)
ax.set_title(
    'Attribution Stability Under Small Perturbations\n'
    '(Higher = less sensitive to input noise)'
)
for i, v in enumerate(values):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print(f"  Saliency stability:          {stability_results['Saliency']:.3f} — sensitive to noise")
print(f"  SmoothGrad stability:        {stability_results['SmoothGrad']:.3f} — more stable")
print(f"  Integrated Gradients:        {stability_results['Integrated Gradients']:.3f} — most stable")

## 4. The Path from Saliency to Integrated Gradients

Integrated Gradients resolves saliency's saturation problem by integrating gradients along a path from baseline to input. This last visualization shows the gradient landscape along that path — the intuition for why integration is the correct solution.

In [ ]:
# Visualize gradient evolution along the integration path
# This is the conceptual foundation of Integrated Gradients

# Create interpolated images: from black baseline to original
n_steps = 10
baseline = torch.zeros_like(input_tensor)

path_attributions = []
path_gradients = []

for step in range(n_steps + 1):
    alpha = step / n_steps  # 0.0 to 1.0
    # Interpolated image: baseline + alpha * (input - baseline)
    interp = baseline + alpha * (input_tensor - baseline)
    interp_grad = interp.clone().requires_grad_(True)

    # Forward + backward
    output = model(interp_grad)[0, top_class]
    output.backward()

    grad = interp_grad.grad.squeeze(0).abs().mean(0).detach().cpu().numpy()  # (H, W)
    path_gradients.append(grad)

    model.zero_grad()

# Integrated gradient = mean of gradients along path
ig_approx = np.stack(path_gradients).mean(axis=0)

# Visualize the path
n_show = 6
show_steps = np.linspace(0, n_steps, n_show, dtype=int)

fig, axes = plt.subplots(2, n_show, figsize=(3*n_show, 6))
fig.suptitle(
    'Integration Path: Gradient Evolution from Baseline to Input\n'
    'IG ≈ average of all these gradient maps',
    fontsize=13
)

for col, step in enumerate(show_steps):
    alpha = step / n_steps

    # Row 0: Interpolated image
    interp_disp = denormalize(
        (baseline + alpha * (input_tensor - baseline)).squeeze(0).cpu()
    ).permute(1, 2, 0).numpy()
    axes[0, col].imshow(interp_disp)
    axes[0, col].set_title(f'α = {alpha:.1f}', fontsize=10)
    axes[0, col].axis('off')

    # Row 1: Gradient at this step
    g = path_gradients[step]
    axes[1, col].imshow(norm99(g), cmap='hot', vmin=0, vmax=1)
    axes[1, col].set_title(f'Gradient at α={alpha:.1f}', fontsize=10)
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

# Show: IG is the mean of all gradients
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(img_np)
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(norm99(path_gradients[-1]), cmap='hot')
axes[1].set_title('Saliency (gradient at input only)\nNoisy, may miss saturated regions')
axes[1].axis('off')

axes[2].imshow(norm99(ig_approx), cmap='hot')
axes[2].set_title(f'IG Approximation (mean of {n_steps+1} gradients)\nSmoother, resolves saturation')
axes[2].axis('off')

plt.suptitle('From Saliency to Integrated Gradients', fontsize=13)
plt.tight_layout()
plt.show()

print("\nKey insight:")
print("Saliency = gradient at a single point (noisy, misses saturated regions)")
print("IG = integral of gradients along path from baseline to input")
print("IG is the theoretically correct generalization of saliency.")
print("Module 02 covers the full derivation and axiomatic justification.")

## Summary

### What You Demonstrated

1. **Saturation failure:** Saliency gives zero attribution to relevant features when ReLU is in its inactive region — this is not a numerical issue but a fundamental limitation
2. **Gradient noise:** Saliency maps change significantly under small input perturbations (low stability correlation)
3. **SmoothGrad** improves stability by averaging over noisy samples, but does not fix saturation
4. **Integrated Gradients** is both more stable and resolves saturation by integrating along the full path
5. **The integration path** shows how IG accumulates gradient information across all activation regimes, not just at the input point

### The Logical Argument for Integrated Gradients

- Saliency measures gradient at one point → noisy, misses saturated regions
- SmoothGrad averages over noisy points → reduces noise but still at the input scale
- Integrated Gradients integrates over the path from baseline to input → correct, stable, satisfies axioms

### What's Next

**Module 02** provides the complete mathematical derivation of Integrated Gradients, explains the axiomatic justification (sensitivity + implementation invariance), and covers the critical baseline choice problem.